# Setup

In [ ]:
import copy
import time
import pickle
import numpy as np
import pandas as pd
import io
import contextlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from scipy.special import expit
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import TSNE
from scipy.special import expit
from scipy import stats
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, average_precision_score, brier_score_loss
import matplotlib.pyplot as plt

In [ ]:
RAND_STATE = 42

In [ ]:
# Load the normalized data
output_dir = Path('../processed/')
output_dir = Path("/kaggle/input/datasets/brocklarson/normalized/") # TODO: change back after done with kaggle

data = np.load(output_dir / 'normalized_split.npz')
X_train_norm = data['X_train']
X_val_norm = data['X_val']
X_test_norm = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']

# with open(output_dir / 'norm_params.pkl', 'rb') as f_in:
#     norm_params = pickle.load(f_in)

print("Pre-processed data loaded successfully")

In [ ]:
for name, X, y in [("TRAINING", X_train_norm, y_train),
                   ("VALIDATION", X_val_norm, y_val),
                   ("TEST", X_test_norm, y_test)]:
    n_sep = int((y == 1).sum())
    n_nsep = int((y == 0).sum())

    print("================")
    print(f"{name} DATA")
    print("================")
    print(f"X shape:    {X.shape}")
    print(f"y shape:    {y.shape}")
    print(f"NSEP count: {n_nsep}")
    print(f"SEP count:  {n_sep}")
    print(f"Ratio:      {n_nsep / n_sep:.2f}:1")
    print()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
torch.backends.cudnn.benchmark = True 

X_train_t = torch.as_tensor(X_train_norm, dtype=torch.float32, device=device)
X_val_t = torch.as_tensor(X_val_norm,   dtype=torch.float32, device=device)
X_test_t = torch.as_tensor(X_test_norm,  dtype=torch.float32, device=device)

y_train_t = torch.as_tensor(y_train, dtype=torch.long, device=device)
y_val_t = torch.as_tensor(y_val, dtype=torch.long, device=device)
y_test_t = torch.as_tensor(y_test, dtype=torch.long, device=device)


# Evaluation Metrics

In [ ]:
def true_skill_statistic(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    # Guard against divide by zero
    if (tp + fn == 0 or
        tn + fp == 0):
        return 0.0

    tss = tp / (tp + fn) - fp / (tn + fp)
    
    return float(tss)

def heidke_skill_score(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    numerator = 2 * ((tp * tn) - (fp * fn))
    denominator = ((tp + fn) * (fn + tn)) + ((tn + fp) * (tp + fp))

    # Guard against divide by zero
    if denominator == 0:
        return 0.0
        
    return float(numerator / denominator)

def brier_skill_score(y_true, y_prob):
    base_rate = np.mean(y_true)
    brier_ref = brier_score_loss(y_true, np.full(len(y_true), base_rate))
    return 1 - brier_score_loss(y_true, y_prob) / brier_ref

def evaluate_model_metrics( 
        y_actual, 
        y_pred, 
        y_prob, 
        model_name="Model"
    ):    
    conf_matrix = confusion_matrix(y_actual, y_pred)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_actual, y_pred),
        "Precision": precision_score(y_actual, y_pred, zero_division=0),
        "Recall": recall_score(y_actual, y_pred, zero_division=0),
        "F1-Score": f1_score(y_actual, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_actual, y_prob),
        "PR-AUC": average_precision_score(y_actual, y_prob),
        "Brier": brier_score_loss(y_actual, y_prob),
        "BSS" : brier_skill_score(y_actual, y_prob),
        "TSS": true_skill_statistic(conf_matrix),
        "HSS2": heidke_skill_score(conf_matrix),
        "Confusion Matrix": conf_matrix
    }
    
    return metrics

def print_metrics(metrics):
    print("===================")
    print("PERFORMANCE METRICS")
    print("===================")
    for name, val in metrics.items():
        formatted_val = round(val,5) if isinstance(val, (float, np.floating)) else val   
        print(f"{name:<17}: {formatted_val}")
    print()

# InfoNCE

### Loss Function

In [ ]:
def sim_clr_loss(z1, z2, temperature=0.1):
    B = z1.shape[0]
    z = torch.cat([z1, z2])                                   # (2B, D)
    sim = z @ z.T / temperature                               # (2B, 2B) cosine similarity / temperature
    sim = sim.masked_fill(torch.eye(2 * B, dtype=torch.bool, device=z.device), float("-inf")) 
    targets = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)             
    return F.cross_entropy(sim, targets)

In [ ]:
def supcon_loss(z1, z2, y, temperature=0.1):
    z = torch.cat([z1, z2])                                    # (2B, D)
    labels = torch.cat([y, y])                                 # (2B,)
    sim = z @ z.T / temperature                                # (2B, 2B)
    self_mask = torch.eye(len(z), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(self_mask, float("-inf"))            

    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True) # log-softmax over all other windows
    positives = (labels[:, None] == labels[None, :]) & ~self_mask
    log_prob = log_prob.masked_fill(~positives, 0)             # keep only the positives

    return -(log_prob.sum(dim=1) / positives.sum(dim=1)).mean()

### Augmentation

In [ ]:
def augment(x, noise_std=0.1, scale_std=0.1):
    # x has shape (batch, time, channels)
    noise = noise_std * torch.randn_like(x)
    scale = 1 + scale_std * torch.randn(x.shape[0], 1, x.shape[2], device=x.device)  # one factor per window per channel
    return x * scale + noise

### GRU Encoder

In [ ]:
class GRUEncoder(nn.Module):
    def __init__(self, in_ch, hidden_ch=64, emb_dim=32):
        super().__init__()
        self.gru = nn.GRU(in_ch, hidden_ch, batch_first=True)
        self.linear = nn.Linear(hidden_ch, emb_dim)

    def forward(self, x):
        _, h = self.gru(x)            
        out = self.linear(h[-1])      
        return F.normalize(out, dim=1)

### Embedding Extraction

In [ ]:
@torch.no_grad()
def embed(enc, X, batch_size=512):
    enc.eval()
    # Run through in sizes of batch_size at a time
    chunks = [enc(X[i:i + batch_size]) for i in range(0, len(X), batch_size)]
    return torch.cat(chunks).cpu().numpy()

### Batching

In [ ]:
def sample_batch_indices(y, batch_size, rng, sep_fraction=0.25):
    sep_idx = np.flatnonzero(y == 1)
    nsep_idx = np.flatnonzero(y == 0)

    n_sep = int(round(batch_size * sep_fraction))
    n_nsep = batch_size - n_sep
    assert n_sep <= len(sep_idx), "sep_fraction too high: would need repeated SEP windows in a batch"

    sep = rng.choice(sep_idx, n_sep, replace=False)
    nsep = rng.choice(nsep_idx, n_nsep, replace=False)
    return np.concatenate([sep, nsep])

### Classifier Heads

In [ ]:
def pairwise_dist(A, B):
    # Euclidean distance between every row of A (n, d) and every row of B (m, d) -> (n, m).
    a2 = (A ** 2).sum(axis=1)[:, None]
    b2 = (B ** 2).sum(axis=1)[None, :]
    d2 = np.maximum(a2 + b2 - 2.0 * (A @ B.T), 0.0)
    return np.sqrt(d2)

In [ ]:
class Model:
    def fit(self, Z, y):
        self._fit(Z, y)
        raw = self._raw_train(Z)                       
        self._mu, self._sd = raw.mean(), raw.std() + 1e-12
        return self
    def score(self, Z): 
        return expit((self._raw(Z) - self._mu) / self._sd)
    def score_train(self, Z): 
        return expit((self._raw_train(Z) - self._mu) / self._sd)
    def _raw_train(self, Z):  
        return self._raw(Z)      

class LogRegModel(Model):
    def __init__(self, C=1.0, class_weight="balanced"):
        lr = LogisticRegression(C=C, class_weight=class_weight, max_iter=2000)
        self.model = lr
    def _fit(self, Z, y):     
        self.model.fit(Z, y)
    def _raw(self, Z):        
        return self.model.decision_function(Z)
    def score(self, Z):       
        return self.model.predict_proba(Z)[:, 1]   
    def score_train(self, Z): 
        return self.score(Z)

class CentroidModel(Model):
    def __init__(self, use_nsep=True):
        self.use_nsep = use_nsep
    def _fit(self, Z, y):
        self.mu_sep  = Z[y == 1].mean(axis=0)
        self.mu_nsep = Z[y == 0].mean(axis=0)
    def _raw(self, Z):
        d_sep = np.linalg.norm(Z - self.mu_sep, axis=1)
        if not self.use_nsep:
            return -d_sep
        return np.linalg.norm(Z - self.mu_nsep, axis=1) - d_sep

class KnnModel(Model):
    def __init__(self, k=5):
        self.k = k
    def _fit(self, Z, y):
        self.sep_pos = np.flatnonzero(y == 1)
        self.Z_sep = Z[self.sep_pos]
    def _raw(self, Z):
        d = pairwise_dist(Z, self.Z_sep)
        return -np.sort(d, axis=1)[:, :self.k].mean(axis=1)
    def _raw_train(self, Z):
        d = pairwise_dist(Z, self.Z_sep)
        d[self.sep_pos, np.arange(len(self.sep_pos))] = np.inf 
        return -np.sort(d, axis=1)[:, :self.k].mean(axis=1)

class SvmModel(Model):
    def __init__(self, kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"):
        self.model = SVC(kernel=kernel, C=C, gamma=gamma, class_weight=class_weight, cache_size=2000)
    def _fit(self, Z, y):  
        self.model.fit(Z, y)
    def _raw(self, Z):     
        return self.model.decision_function(Z)

class XgbModel(Model):
    def __init__(self, scale_pos_weight="balanced", max_depth=3, min_child_weight=5, n_estimators=200, learning_rate=0.05, subsample=0.7, colsample_bytree=0.5, reg_lambda=10, random_state=RAND_STATE):
        self.scale_pos_weight = scale_pos_weight
        self.params = dict(
            max_depth=max_depth, 
            min_child_weight=min_child_weight,
            n_estimators=n_estimators, 
            learning_rate=learning_rate,
            subsample=subsample, 
            colsample_bytree=colsample_bytree,
            reg_lambda=reg_lambda, 
            random_state=random_state, 
            n_jobs=-1
        )
    def _fit(self, Z, y):
        y = np.asarray(y)
        spw = (y == 0).sum() / max((y == 1).sum(), 1) if self.scale_pos_weight == "balanced" else self.scale_pos_weight
        self.model = XGBClassifier(scale_pos_weight=spw, **self.params)
        self.model.fit(Z, y)
    def _raw(self, Z):
        return self.model.predict(Z, output_margin=True)
    def score(self, Z):
        return self.model.predict_proba(Z)[:, 1]
    def score_train(self, Z):
        return self.score(Z)

HEADS = {"logreg": LogRegModel, "centroid": CentroidModel, "knn": KnnModel, "svm": SvmModel, "xgb": XgbModel}

def make_model(name, params=None):
    return HEADS[name](**(params or {}))

### Classifier Evaluation

In [ ]:
def evaluate_classifier(head, head_params, Z_train, y_train, Z_val, y_val, Z_eval=None, y_eval=None, model_name="Model"):
    # Instantiate the model using factory function and fit on training embeddings
    clf = make_model(head, head_params).fit(Z_train, y_train)

    # Choose the threshold on validation
    s_train, s_val = clf.score_train(Z_train), clf.score(Z_val)
    fpr, tpr, thr = roc_curve(y_val, s_val)
    threshold = float(thr[np.argmax(tpr - fpr)])

    # Evaluate on the requested set (validation by default)
    if Z_eval is None:
        y_eval, s_eval = y_val, s_val
    else:
        s_eval = clf.score(Z_eval)

    # Evaluate
    y_pred = (s_eval >= threshold).astype(int)
    metrics = evaluate_model_metrics(y_eval, y_pred, s_eval, model_name=model_name)

    # Compute extra training metrics
    fpr_t, tpr_t, _ = roc_curve(y_train, s_train)
    metrics["train_ROC_AUC"] = roc_auc_score(y_train, s_train)
    metrics["train_TSS"] = float(np.max(tpr_t - fpr_t))
    metrics["Threshold"] = threshold
    return metrics

### Encoder Training

In [ ]:
def evaluate_step(step, train_loss, enc, X_train_t, y_train, X_val_t, y_val, head, head_params):
    Z_train, Z_val = embed(enc, X_train_t), embed(enc, X_val_t)
    results = evaluate_classifier(head, head_params, Z_train, y_train, Z_val, y_val)
    enc.train()
    return {"step": step, "train_loss": train_loss, "TSS": results["TSS"], "train_TSS": results["train_TSS"], "ROC_AUC": results["ROC-AUC"], "train_ROC_AUC": results["train_ROC_AUC"]}

def print_step_metrics(metrics):
    print(f"step {metrics['step']:5d} | loss {metrics['train_loss']:.4f} | Train TSS {metrics['train_TSS']:.3f} | Val TSS {metrics['TSS']:.3f} | Train ROC-AUC {metrics['train_ROC_AUC']:.3f} | Val ROC-AUC {metrics['ROC_AUC']:.3f}")

In [ ]:
def train_encoder(X_train_t, y_train, X_val_t, y_val,
                  seed=RAND_STATE,          # Random state seed
                  n_steps=1000,             # Training steps (0 = untrained reference)
                  batch_size=256,           # Windows per step -> 2 * batch_size views
                  lr=1e-3,                  # Learning rate
                  weight_decay=1e-4,        # L2 penalty: shrinks weights toward 0
                  hidden_ch=64,             # GRU size
                  emb_dim=32,               # Embedding dimension
                  temperature=0.1,          # Loss temperature
                  sep_fraction=0.25,        # Portion of each batch drawn from SEP windows
                  noise_std=0.1,            # Augmentation strength: jitter
                  scale_std=0.1,            # Augmentation strength: per-channel scaling
                  eval_every=100,           # How often to evaluate
                  loss_type="infonce",      # "infonce" (no labels in the loss) or "supcon" (labels in the loss)
                  head="logreg",            # Which classifier the printed curves use
                  head_params=None,         # Classifier's settings
                  ):

    if loss_type not in ("infonce", "supcon"):
        raise ValueError(f"No loss found matching '{loss_type}'")

    device = X_train_t.device
    torch.manual_seed(seed)                 # controls the encoder's starting weights + augmentation noise
    rng = np.random.default_rng(seed)       # controls which windows are sampled
    y_np = np.asarray(y_train)
    y_t = torch.as_tensor(y_np, dtype=torch.long, device=device)

    # Initialize encoder and optimizer
    enc = GRUEncoder(in_ch=X_train_t.shape[2], hidden_ch=hidden_ch, emb_dim=emb_dim).to(device)
    optimizer = torch.optim.Adam(enc.parameters(), lr=lr, weight_decay=weight_decay)

    # Evaluate untrained model (step 0)
    history = [evaluate_step(0, float("nan"), enc, X_train_t, y_train, X_val_t, y_val, head, head_params)]
    print_step_metrics(history[0])

    recent_losses = []
    for step in range(1, n_steps + 1):
        # Pick a batch of windows
        idx = torch.as_tensor(sample_batch_indices(y_np, batch_size, rng, sep_fraction), dtype=torch.long, device=device)
        x = X_train_t[idx]

        # Two different views of the same windows -> positive pairs
        z1 = enc(augment(x, noise_std, scale_std))
        z2 = enc(augment(x, noise_std, scale_std))

        if loss_type == "infonce":
            loss = sim_clr_loss(z1, z2, temperature)
        elif loss_type == "supcon":
            loss = supcon_loss(z1, z2, y_t[idx], temperature)
        else:
            raise ValueError(f"Loss type of {loss_type} not recognized")

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        recent_losses.append(loss.item())

        # Evaluate on validation every `eval_every` steps, and at the very end
        if step % eval_every == 0 or step == n_steps:
            history.append(evaluate_step(step, float(np.mean(recent_losses)), enc, X_train_t, y_train, X_val_t, y_val, head, head_params))
            recent_losses = []
            print_step_metrics(history[-1])

    return enc, history

## Experiments

### Experiment Runner

In [ ]:
RESULTS = {} # Stores all run results for later comparison

In [ ]:
def run_infonce(cfg, seed, X_eval_t, y_eval):
    # Train encoder on train, fit head on train, pick threshold on val, score on the eval set.
    enc, _ = train_encoder(X_train_t, y_train, X_val_t, y_val, seed=seed, **cfg)
    Z_train = embed(enc, X_train_t)
    Z_val = embed(enc, X_val_t)
    Z_eval = embed(enc, X_eval_t)
    return evaluate_classifier(cfg["head"], cfg["head_params"], Z_train, y_train, Z_val, y_val, Z_eval, y_eval, model_name=f"{cfg['loss_type']}+{cfg['head']} seed {seed}")

def run_experiment(name, grid, n_search_seeds, n_final_runs, select_metric):
    print("═" * 70)
    print(f"{name}: searching {len(grid)} configs x {n_search_seeds} seeds on validation")
    print("═" * 70)

    # Loop through configs and seeds. Train and eval on validation
    rows = []
    for c, cfg in enumerate(grid):
        for i in range(n_search_seeds):
            seed = RAND_STATE + i
            # Print run header
            header_label = f"\n[config {c+1}/{len(grid)} | seed {seed}] {cfg}"
            print(header_label)

            # Run training and eval
            t0 = time.perf_counter()
            m = run_infonce(cfg, seed, X_val_t, y_val)
            secs = time.perf_counter() - t0

            # Add to df and print results
            row_data = {"config": c, "seed": seed}
            row_data.update(m)
            rows.append(row_data)
            print(f"Results: Val TSS={m['TSS']:.4f} | ROC-AUC={m['ROC-AUC']:.4f} | Thr={m['Threshold']:.4f} | {secs:.0f}s")

    search_df = pd.DataFrame(rows)
    summary = search_df.groupby("config")[["TSS", "ROC-AUC"]].agg(["mean", "std"]).round(3)
    print(summary.rename(index=lambda c: c + 1))

    best_c = int(search_df.groupby("config")[select_metric].mean().idxmax())
    best_cfg = grid[best_c]
    print(f"\nBest by val {select_metric}: config #{best_c+1} | {best_cfg}")

    # One-time evaluation on test
    print(f"\nRunning {n_final_runs} final runs on test")
    print("─" * 70)
    final_rows = []
    for run in range(n_final_runs):
        seed = RAND_STATE + 100 + run
        print(f"\nRun {run+1} (seed {seed})", flush=True)
        
        t0 = time.perf_counter()
        m = run_infonce(best_cfg, seed, X_test_t, y_test)
        secs = time.perf_counter() - t0 

        row_data = {"config": best_c, "seed": seed}
        row_data.update(m)
        final_rows.append(row_data)
        print_metrics(m)

    final_df = pd.DataFrame(final_rows)
    print(f"{name} test: TSS {final_df['TSS'].mean():.3f} ± {final_df['TSS'].std():.3f} | AUC {final_df['ROC-AUC'].mean():.3f} ± {final_df['ROC-AUC'].std():.3f}")
    return {"name": name, "best_cfg": grid[best_c], "search": search_df, "final": final_df}


### Config

In [ ]:
BASE_CONFIG = dict(
    # loss
    temperature=0.1,

    # augmentation
    noise_std=0.1,
    scale_std=0.1,

    # encoder
    hidden_ch=64,
    emb_dim=32,

    # batches
    batch_size=256,
    sep_fraction=0.25,

    # training
    lr=1e-3,
    weight_decay=1e-4,
    n_steps=1000,
    eval_every=1000,

    # head specific parameters
    head="logreg",
    head_params={},
)

### Logistic Regression

In [ ]:
LOGREG_BASE = {**BASE_CONFIG, "head": "logreg"}

rng = np.random.default_rng(RAND_STATE + 1)       
head_rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []
for i in range(15):
    cfg = {**LOGREG_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-5, -3), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-5, -2), 6)
    cfg["temperature"] = float(rng.choice([0.1, 0.2, 0.5, 1.0]))
    cfg["hidden_ch"] = int(rng.choice([16, 32, 64]))
    cfg["emb_dim"] = int(rng.choice([16, 32]))
    cfg["batch_size"] = int(rng.choice([128, 256]))
    cfg["sep_fraction"] = round(rng.uniform(0.1, 0.45), 3)
    cfg["noise_std"] = float(rng.choice([0.05, 0.1, 0.2]))
    cfg["scale_std"] = float(rng.choice([0.05, 0.1, 0.2]))
    cfg["head_params"] = {"C": round(10 ** head_rng.uniform(-3, 0.5), 5)}
    RANDOM_CFGS.append(cfg)

LOGREG_GRID = [
    *[{**c, "loss_type": "infonce", "n_steps": 0} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "infonce"} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "supcon"} for c in RANDOM_CFGS],
]

RESULTS[f"logreg"] = run_experiment(
    name=f"Logistic Regression",
    grid=LOGREG_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

### Centroid

In [ ]:
CENTROID_BASE = {**BASE_CONFIG, "head": "centroid", "head_params": {}}

rng = np.random.default_rng(RAND_STATE + 1)       
head_rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []
for i in range(15):
    cfg = {**CENTROID_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-5, -3), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-5, -2), 6)
    cfg["temperature"] = float(rng.choice([0.1, 0.2, 0.5, 1.0]))
    cfg["hidden_ch"] = int(rng.choice([16, 32, 64]))
    cfg["emb_dim"] = int(rng.choice([16, 32]))
    cfg["batch_size"] = int(rng.choice([128, 256]))
    cfg["sep_fraction"] = round(rng.uniform(0.1, 0.45), 3)
    cfg["noise_std"] = float(rng.choice([0.05, 0.1, 0.2]))
    cfg["scale_std"] = float(rng.choice([0.05, 0.1, 0.2]))
    RANDOM_CFGS.append(cfg)

CENTROID_GRID = [
    *[{**c, "loss_type": "infonce", "n_steps": 0} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "infonce"} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "supcon"} for c in RANDOM_CFGS],
]

RESULTS[f"centroid"] = run_experiment(
    name=f"Centroid",
    grid=CENTROID_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

### SVM

In [ ]:
SVM_BASE = {**BASE_CONFIG, "head": "svm"}

GAMMA_OPTIONS = ["scale", 0.03, 0.1, 0.3, 1.0]

rng = np.random.default_rng(RAND_STATE + 1)       
head_rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []
for i in range(15):
    cfg = {**SVM_BASE}
    cfg["lr"]=round(10 ** rng.uniform(-4, -2.5), 6)
    cfg["weight_decay"]=round(10 ** rng.uniform(-5, -2), 6)
    cfg["temperature"]=float(rng.choice([0.05, 0.1, 0.2, 0.5]))
    cfg["hidden_ch"]=int(rng.choice([32, 64, 128]))
    cfg["emb_dim"]=int(rng.choice([16, 32]))
    cfg["batch_size"]=int(rng.choice([128, 256]))
    cfg["sep_fraction"] = round(rng.uniform(0.1, 0.45), 3)
    cfg["noise_std"]=float(rng.choice([0.05, 0.1, 0.2]))
    cfg["scale_std"]=float(rng.choice([0.05, 0.1, 0.2]))
    cfg["head_params"] = {
        "kernel": "rbf",
        "C": round(10 ** head_rng.uniform(-3, 0.5), 5),
        "gamma": GAMMA_OPTIONS[int(head_rng.integers(len(GAMMA_OPTIONS)))],
    }
    RANDOM_CFGS.append(cfg)

SVM_GRID = [
    *[{**c, "loss_type": "infonce", "n_steps": 0} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "infonce"} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "supcon"} for c in RANDOM_CFGS],
]


RESULTS[f"SVM"] = run_experiment(
    name=f"SVM",
    grid=SVM_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

### kNN

In [ ]:
KNN_BASE = {**BASE_CONFIG, "head": "knn"}

rng = np.random.default_rng(RAND_STATE + 1)        
head_rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []
for i in range(15):
    cfg = {**KNN_BASE}
    cfg["lr"]=round(10 ** rng.uniform(-4, -2.5), 6)
    cfg["weight_decay"]=round(10 ** rng.uniform(-5, -2), 6)
    cfg["temperature"]=float(rng.choice([0.05, 0.1, 0.2, 0.5]))
    cfg["hidden_ch"]=int(rng.choice([32, 64, 128]))
    cfg["emb_dim"]=int(rng.choice([16, 32]))
    cfg["batch_size"]=int(rng.choice([128, 256]))
    cfg["sep_fraction"] = round(rng.uniform(0.1, 0.45), 3)
    cfg["noise_std"]=float(rng.choice([0.05, 0.1, 0.2]))
    cfg["scale_std"]=float(rng.choice([0.05, 0.1, 0.2]))
    cfg["head_params"] = {"k": int(head_rng.choice([20, 40, 60, 80, 100]))}
    RANDOM_CFGS.append(cfg)

KNN_GRID = [
    *[{**c, "loss_type": "infonce", "n_steps": 0} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "infonce"} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "supcon"} for c in RANDOM_CFGS],
]


RESULTS[f"kNN"] = run_experiment(
    name=f"kNN",
    grid=KNN_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

### XGBOOST

In [ ]:
XGB_BASE = {**BASE_CONFIG, "head": "xgb"}

rng = np.random.default_rng(RAND_STATE + 1)        # same encoder configs for every classifier
head_rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []
for i in range(15):
    cfg = {**XGB_BASE}
    cfg["lr"]=round(10 ** rng.uniform(-4, -2.5), 6)
    cfg["weight_decay"]=round(10 ** rng.uniform(-5, -2), 6)
    cfg["temperature"]=float(rng.choice([0.05, 0.1, 0.2, 0.5]))
    cfg["hidden_ch"]=int(rng.choice([32, 64, 128]))
    cfg["emb_dim"]=int(rng.choice([16, 32]))
    cfg["batch_size"]=int(rng.choice([128, 256]))
    cfg["sep_fraction"] = round(rng.uniform(0.1, 0.45), 3)
    cfg["noise_std"]=float(rng.choice([0.05, 0.1, 0.2]))
    cfg["scale_std"]=float(rng.choice([0.05, 0.1, 0.2]))
    cfg["head_params"] = {
        "max_depth": int(rng.choice([2, 3, 4, 6])),
        "min_child_weight": int(rng.choice([10, 20, 40, 800])),
        "n_estimators": int(rng.choice([50, 100, 200])),
        "learning_rate": round(10 ** rng.uniform(-2.5, -1), 4),
        "colsample_bytree": float(rng.choice([0.3, 0.5])),
    }
    RANDOM_CFGS.append(cfg)

XGB_GRID = [
    *[{**c, "loss_type": "infonce", "n_steps": 0} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "infonce"} for c in RANDOM_CFGS],
    *[{**c, "loss_type": "supcon"} for c in RANDOM_CFGS],
]


RESULTS[f"xgb"] = run_experiment(
    name=f"XGBoost",
    grid=XGB_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

# Ablation Test

In [ ]:
# Full config chosen because it was the best of above random search
FULL_CFG = {'temperature': 1.0, 'noise_std': 0.2, 'scale_std': 0.05, 'hidden_ch': 32, 'emb_dim': 16, 'batch_size': 128, 'sep_fraction': 0.412, 'lr': 3.4e-05, 'weight_decay': 0.000182, 'n_steps': 1000, 'eval_every': 1000, 'head': 'logreg', 'head_params': {'C': 0.00142}, 'loss_type': 'supcon'}
ABL_SEEDS = [RAND_STATE + 200 + i for i in range(8)]

# Various Ablation Test Configs
COMPONENT_ABLATIONS = {
    "full":            {},
    "untrained":       {"n_steps": 0},
    "loss=infonce":    {"loss_type": "infonce"},                  
    "no_noise_aug":    {"noise_std": 0.0},
    "no_scale_aug":    {"scale_std": 0.0},
    "no_aug":          {"noise_std": 0.0, "scale_std": 0.0},
    "natural_batches": {"sep_fraction": float(y_train.mean())}, 
    "no_weight_decay": {"weight_decay": 0.0},
}
HYPERPARAMETERS_ABLATIONS = {
    "temperature=0.1": {"temperature": 0.1},
    "temperature=0.5": {"temperature": 0.5},
    "temperature=2.0": {"temperature": 2.0},
    "hidden_ch=16":    {"hidden_ch": 16},
    "hidden_ch=64":    {"hidden_ch": 64},
    "emb_dim=32":      {"emb_dim": 32},
    "batch_size=256":  {"batch_size": 256},
}
LR_ABLATIONS = {
    "lr x10":   {"lr": 3.4e-04},
    "lr x100":  {"lr": 3.4e-03},
    "steps x3": {"n_steps": 3000},
}

In [ ]:
def run_variants(variants, base_cfg=FULL_CFG, seeds=ABL_SEEDS, X_eval=None, y_eval=None, test_name=""):
    X_eval = X_val_t if X_eval is None else X_eval
    y_eval = y_val if y_eval is None else y_eval
    rows = []
    for name, overrides in variants.items():
        cfg = {**base_cfg, **overrides}
        for seed in seeds:
            t0 = time.perf_counter()
            with contextlib.redirect_stdout(io.StringIO()):          # no per-step prints
                m = run_infonce(cfg, seed, X_eval, y_eval)
            rows.append({"variant": name, "seed": seed, "TSS": m["TSS"], "ROC-AUC": m["ROC-AUC"],
                         "train_TSS": m["train_TSS"]})
            print(f"{test_name}{name:<20} seed {seed} | TSS {m['TSS']:.3f} | AUC {m['ROC-AUC']:.3f} | {time.perf_counter()-t0:.0f}s", flush=True)
    return pd.DataFrame(rows)

def paired_summary(df, ref="full", metric="TSS"):
    wide = df.pivot(index="seed", columns="variant", values=metric)
    out = []
    for v in wide.columns:
        d = (wide[v] - wide[ref]).dropna()
        n = len(d)
        sem = d.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
        ci = stats.t.ppf(0.975, n - 1) * sem if n > 1 else np.nan
        out.append({"variant": v, f"{metric} mean": wide[v].mean(), f"{metric} std": wide[v].std(ddof=1),
                    "diff vs ref": d.mean(), "95% CI ±": ci,
                    "seeds better": int((d > 0).sum()), "n": n,
                    "distinguishable": bool(n > 1 and abs(d.mean()) > ci)})
    return pd.DataFrame(out).set_index("variant").round(3)

In [ ]:
# ==== LEAVE ONE OUT TESTS ====
# Component Ablation Test
df_comp = run_variants(COMPONENT_ABLATIONS, test_name="[component] ")
print("\n=== COMPONENT ABLATION ===")
print(paired_summary(df_comp, metric="TSS"))
print(paired_summary(df_comp, metric="ROC-AUC"))

# Hyperparamerter Ablation Test
df_sens = run_variants({"full": {}, **HYPERPARAMETERS_ABLATIONS}, test_name="[hyperparameters] ")
print("\n=== HYPERPARAMETER ABLATION ===")
print(paired_summary(df_sens, metric="TSS"))

# Learning Rate Ablation Test
df_diag = run_variants({"full": {}, "untrained": {"n_steps": 0}, **LR_ABLATIONS}, test_name="[learning rate] ")
print("\n=== LEARNING RATE ===")
print(paired_summary(df_diag, metric="TSS"))

# === ADD BACK IN TESTS ===
STRIPPED = {"noise_std": 0.0, "scale_std": 0.0, "sep_fraction": float(y_train.mean()), "weight_decay": 0.0}
ADD_BACK = {
    "stripped":          {},
    "+noise_aug":        {"noise_std": FULL_CFG["noise_std"]},
    "+scale_aug":        {"scale_std": FULL_CFG["scale_std"]},
    "+balanced_batches": {"sep_fraction": FULL_CFG["sep_fraction"]},
    "+weight_decay":     {"weight_decay": FULL_CFG["weight_decay"]},
    "all":               {k: FULL_CFG[k] for k in STRIPPED},
}
df_add = run_variants(ADD_BACK, base_cfg={**FULL_CFG, **STRIPPED}, test_name="[add-back-in] ")
print("\n=== ADD-BACK ===")
print(paired_summary(df_add, ref="stripped", metric="TSS"))

In [ ]:
# Final test on test set for final report
TEST_CFGS = {
    "full":        {},
    "untrained":   {"n_steps": 0},
    "no_aug":      {"noise_std": 0.0, "scale_std": 0.0},  
    "natural_batches": {"sep_fraction": float(y_train.mean())},
}
TEST_SEEDS = [RAND_STATE + 300 + i for i in range(5)]
df_test = run_variants(TEST_CFGS, seeds=TEST_SEEDS, X_eval=X_test_t, y_eval=y_test, test_name="[FINAL TEST] ")
print(paired_summary(df_test, metric="TSS"))

# t-SNE Visualization

In [ ]:
# Set to desired config and seed
SEED = 142
tsne_cfg = {'temperature': 1.0, 'noise_std': 0.2, 'scale_std': 0.05, 'hidden_ch': 32, 'emb_dim': 16, 'batch_size': 128, 'sep_fraction': 0.412, 'lr': 3.4e-05, 'weight_decay': 0.000182, 'n_steps': 1000, 'eval_every': 1000, 'head': 'logreg', 'head_params': {'C': 0.00142}, 'loss_type': 'supcon'}

In [ ]:
def train_and_embed(cfg, seed, n_steps=None, label=""):
    # Run encoders and evaluate
    cfg["n_steps"] = n_steps
    print(f"\n=== {label} (n_steps={cfg['n_steps']}) ===")

    enc, _ = train_encoder(X_train_t, y_train, X_val_t, y_val, seed=seed, **cfg)
    Z = {
        "train": embed(enc, X_train_t), 
        "val": embed(enc, X_val_t), 
        "test": embed(enc, X_test_t)
    }
    metrics = evaluate_classifier(cfg["head"], cfg["head_params"], Z["train"], y_train, Z["val"], y_val, Z["test"], y_test, model_name=label)
    return Z, metrics

# Get embeddings and metrics
Z_before, m_before = train_and_embed(tsne_cfg, SEED, n_steps=0, label="Untrained")
Z_after,  m_after = train_and_embed(tsne_cfg, SEED, n_steps=1000, label="Trained")

print(f"\nUntrained TSS={m_before['TSS']:.3f}  ROC-AUC={m_before['ROC-AUC']:.3f}")
print(f"Trained TSS={m_after['TSS']:.3f}  ROC-AUC={m_after['ROC-AUC']:.3f}")

In [ ]:
def run_tsne(Z, perplexity=30, seed=RAND_STATE):
    return TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        metric="euclidean", 
        random_state=seed,
    ).fit_transform(Z)

tsne_before = run_tsne(Z_before["test"])
tsne_after  = run_tsne(Z_after["test"])

In [ ]:
def scatter_tsne(ax, emb2d, y, title):
    nsep = y == 0
    sep = y == 1

    ax.scatter(emb2d[nsep, 0], emb2d[nsep, 1], s=20, c="#2a80db", alpha=0.35, linewidths=0, label=f"NSEP (n={nsep.sum()})", rasterized=True)
    ax.scatter(emb2d[sep, 0], emb2d[sep, 1], s=55, c="#d62728", edgecolors="black", linewidths=0.6, label=f"SEP (n={sep.sum()})", zorder=3)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="best", frameon=True, fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
scatter_tsne(axes[0], tsne_before, y_test, "Untrained")
scatter_tsne(axes[1], tsne_after, y_test, "Trained")
plt.tight_layout()
plt.show()